In [ ]:
import re

def censor_pii(text: str) -> str:

    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '[EMAIL]', text)

    text = re.sub(r'\b\d{16}\b', '[NIK]', text)

    text = re.sub(r'\b(?:\+62|62|08)\d{8,12}\b', '[PHONE]', text)

    text = re.sub(r'\b\d{10,15}\b', '[REKENING]', text)

    return text

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

In [ ]:


MODEL_PATH = './model_IndoBERT CRM classifier'

print(f"Loading model offline from: {MODEL_PATH}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=device)



Loading model offline from: ./indobert_submission...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3270.20it/s]


In [ ]:

def predict_with_privacy(text: str):
    """
    End-to-end inference pipeline:
    1. Intercepts and cleans PII
    2. Feeds clean text to the SLM
    3. Returns results
    """
    safe_text = censor_pii(text)

    prediction = classifier(safe_text)[0]

    return {
        "original_text": text,
        "safe_text": safe_text,
        "prediction_label": prediction['label'],
        "confidence_score": round(prediction['score'], 4)
    }



In [ ]:


test_cases = [
    "Tolong cek pesanan saya, kok belum sampai? Email saya budi.test@gmail.com dan nomor HP 081234567890.",
    "Saya mau refund dana ke rekening BCA 014234567890 atas nama NIK 3201012345678901.",
    "Halo min, nanya dong harga paket yang paling murah berapa ya? Hubungi saya di +6285712341234",
    "Barangnya bagus banget, sesuai ekspektasi!"
]

print("\n--- RUNNING INFERENCE ---")
for i, text in enumerate(test_cases, 1):
    result = predict_with_privacy(text)
    print(f"\nTest Case {i}:")
    print(f"Original : {result['original_text']}")
    print(f"Filtered : {result['safe_text']}")
    print(f"Label    : {result['prediction_label']} (Score: {result['confidence_score']})")


--- RUNNING INFERENCE ---

Test Case 1:
Original : Tolong cek pesanan saya, kok belum sampai? Email saya budi.test@gmail.com dan nomor HP 081234567890.
Filtered : Tolong cek pesanan saya, kok belum sampai? Email saya [EMAIL_CENSORED] dan nomor HP [PHONE_CENSORED].
Label    : Transactional (Score: 0.496)

Test Case 2:
Original : Saya mau refund dana ke rekening BCA 014234567890 atas nama NIK 3201012345678901.
Filtered : Saya mau refund dana ke rekening BCA [REKENING_CENSORED] atas nama NIK [NIK_CENSORED].
Label    : Transitional (Score: 0.997)

Test Case 3:
Original : Halo min, nanya dong harga paket yang paling murah berapa ya? Hubungi saya di +6285712341234
Filtered : Halo min, nanya dong harga paket yang paling murah berapa ya? Hubungi saya di +[PHONE_CENSORED]
Label    : Transitional (Score: 0.9994)

Test Case 4:
Original : Barangnya bagus banget, sesuai ekspektasi!
Filtered : Barangnya bagus banget, sesuai ekspektasi!
Label    : Communal (Score: 0.9839)
